# Figura de taxones por busqueda y direccion

**Entra:** `tabla2_filas_v2.csv` (busqueda A, microalgas) y `tabla1_filas_nuevas.csv` (busqueda B, modelos de EA).

**Sale:** `fig_taxa_by_direction.png`.

La frecuencia cuenta **estudios, no filas**: un estudio que reporta el mismo taxon en dos brazos
tiene que contar una vez. Por eso se usa `nunique()` sobre el identificador del estudio, `pmid`
en la tabla de microalgas y `referencia` en la de EA.

**OJO:** las celdas de definiciones tienen que correrse antes que las que las usan. Si se edita una
funcion, hay que volver a correr su celda; editar no es cargar.


In [1]:
import re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [2]:
VERDE, ROJO = "#2E7D5B", "#A4303F"
MARCA = {"genero": "g", "familia": "f", "filo": "p", "clase": "c", "especie": "sp", "razon": "ratio"}
RANGOS_OK = ["genero", "familia", "filo", "clase", "especie"]   # las razones F/B quedan fuera: no son un taxon


## Rango taxonomico

La tabla de microalgas ya trae columna `nivel`; la de EA no, asi que hay que deducirlo del nombre.
Mezclar rangos sin marcarlos hace que un filo y un genero se lean como comparables, y no lo son.


In [3]:
def rango(t):
    """Infiere el rango taxonomico del nombre. La tabla de microalgas ya trae columna `nivel`;
    la de EA no, asi que hay que deducirlo por sufijo.

    TRAMPA: 'Rikenellaceae_RC9_gut_group' termina en _group y es un GENERO, no una familia,
            por eso la excepcion va antes que la regla de -aceae.
    """
    t = str(t).strip()
    if "/" in t or t.lower().startswith(("razon", "razón", "ratio", "relacion", "relación")):
        return "razon"
    if t.endswith("gut_group") or t.endswith("group"):
        return "genero"
    if re.search(r"\s", t):
        return "especie"
    if t.endswith("aceae"):
        return "familia"
    # TRAMPA: 'Firmicutes' no termina en -etes ni en -ota y caeria en genero. Los filos con
    #         nombre historico van listados a mano.
    if t in ("Firmicutes", "Bacillota", "Bacteroidetes", "Bacteroidota", "Proteobacteria",
             "Actinobacteria", "Actinobacteriota", "Verrucomicrobia", "Verrucomicrobiota",
             "Deferribacteres", "Deferribacterota", "Campilobacterota", "Campylobacterota",
             "Desulfobacterota", "Fusobacteriota", "Patescibacteria", "Tenericutes", "Spirochaetes"):
        return "filo"
    if t.endswith(("ota", "etes", "bacteria")):
        return "filo"
    if t in ("Bacilli", "Bacteroidia", "Clostridia", "Verrucomicrobiae", "Gammaproteobacteria"):
        return "clase"
    return "genero"


## Conteo por taxon y direccion

**OJO:** ordenar por el total, incluyendo los estudios que no declaran direccion, sube taxones que
no dibujan ninguna barra. Se ordena por los que si la declaran y el resto se anota al margen.


In [4]:
def contar(df, col_estudio, pos, neg, inc, n=13):
    """Devuelve etiquetas y conteos de estudios por taxon, ordenados por evidencia con direccion.

    TRAMPA: ordenar por el total (incluyendo los sin direccion) sube taxones que no dibujan
            ninguna barra. Se ordena por pos+neg y los sin direccion se anotan al margen.
    """
    d = df[df["nivel"].isin(RANGOS_OK)]
    P = d[d.direccion == pos].groupby("taxon")[col_estudio].nunique()
    N = d[d.direccion == neg].groupby("taxon")[col_estudio].nunique()
    I = d[d.direccion == inc].groupby("taxon")[col_estudio].nunique()
    con_direccion = P.add(N, fill_value=0)
    taxa = con_direccion[con_direccion > 0].sort_values(ascending=False).head(n).index[::-1]
    nivel = d.groupby("taxon").nivel.first()
    etiquetas = [f"{t.replace('_', ' ')}  ({MARCA[nivel[t]]})" for t in taxa]
    return (etiquetas,
            np.array([P.get(t, 0) for t in taxa]),
            np.array([N.get(t, 0) for t in taxa]),
            np.array([I.get(t, 0) for t in taxa]))


## Dibujo de un panel

In [5]:
def panel(ax, etiquetas, pos, neg, inc, titulo, etq_izq, etq_der):
    y = np.arange(len(etiquetas))
    ax.barh(y, pos, color=VERDE, height=0.68, edgecolor="none")
    ax.barh(y, -neg, color=ROJO, height=0.68, edgecolor="none")
    ax.set_yticks(y)
    ax.set_yticklabels(etiquetas, fontsize=7.4)
    for t in ax.get_yticklabels():
        t.set_fontstyle("italic")
    tope = max(pos.max(), neg.max())
    lim = tope + 1.45                       # margen a la derecha para la nota de "no direction"
    for i, (a, b, c) in enumerate(zip(pos, neg, inc)):
        if a:
            ax.text(a + 0.08, i, str(a), va="center", fontsize=6.6, color=VERDE)
        if b:
            ax.text(-b - 0.08, i, str(b), va="center", ha="right", fontsize=6.6, color=ROJO)
        if c:
            ax.text(lim - 0.05, i, f"+{c} no direction", va="center", ha="right", fontsize=5.9, color="0.5")
    ax.set_xlim(-lim, lim)
    ticks = list(range(-int(tope), int(tope) + 1))
    ax.set_xticks(ticks)
    ax.set_xticklabels([str(abs(v)) for v in ticks], fontsize=7)   # el eje muestra n, no numeros negativos
    ax.axvline(0, color="0.3", lw=0.8)
    ax.set_xlabel("Studies reporting it (n)", fontsize=8)
    ax.set_title(titulo, fontsize=9, loc="left", pad=16)
    ax.text(-lim, len(etiquetas) - 0.15, etq_izq, fontsize=6.8, color=ROJO, ha="left", va="bottom")
    ax.text(lim, len(etiquetas) - 0.15, etq_der, fontsize=6.8, color=VERDE, ha="right", va="bottom")
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.grid(axis="x", lw=0.4, alpha=0.3)
    ax.set_axisbelow(True)
    ax.tick_params(axis="y", length=0)


## Datos

In [6]:
# --- datos -------------------------------------------------------------------
micro = pd.read_csv("tabla2_filas_v2.csv", dtype=str)        # busqueda A
ea = pd.read_csv("tabla1_filas_nuevas.csv", dtype=str)       # busqueda B
ea["nivel"] = ea.taxon.map(rango)                            # la de microalgas ya trae `nivel`

eA, pA, nA, iA = contar(micro, "pmid", "aumenta", "disminuye", "no concluyente")
eB, pB, nB, iB = contar(ea, "referencia", "Positivo", "Negativo", "No concluyente")


## Figura

**OJO:** el eje x muestra `n` a los dos lados del cero, nunca numeros negativos. El signo lo da el
color y la etiqueta de cada lado, no el numero.


In [12]:
# --- figura ------------------------------------------------------------------
fig, axs = plt.subplots(1, 2, figsize=(11.2, 5.9))
panel(axs[0], eA, pA, nA, iA, "A. Search A: eukaryotic microalgae (Table 5)",
      "decreased by the intervention  \u2190", "\u2192  increased by the intervention")
panel(axs[1], eB, pB, nB, iB, "B. Search B: murine AD models (Table 6)",
      "associated with pathology  \u2190", "\u2192  associated with protection"),
fig.tight_layout(rect=[0, 0.05, 1, 0.945])
fig.savefig("fig_taxa_by_direction.png", dpi=600, bbox_inches="tight")

# fig.text(0.012, 0.025,
  #       "Rank in parentheses: g genus \u00b7 f family \u00b7 p phylum \u00b7 c class. Taxa are ordered by the number of studies "
    #      "with a declared direction; studies without one are noted at the margin. The two scales are not equivalent: in A the "
    #     "direction is the change in abundance with the intervention, in B it is the association each study interprets.",
     #    fontsize=6.6, color="0.3", ha="left")

# fig.text(0.012, 0.975, "Most frequently reported taxa in each search, by direction of the reported change",
         # fontsize=10.5, ha="left", va="top")


## Control: conteos que quedaron en cada panel

In [10]:
for etq, p, n, i in [(eA, pA, nA, iA), (eB, pB, nB, iB)]:
    print(" | ".join(f"{e.split(chr(40))[0].strip()}: +{a}/-{b}" + (f" ({c} sin direccion)" if c else "")
                     for e, a, b, c in zip(etq, p, n, i)))
    print()


Parabacteroides: +0/-2 (1 sin direccion) | Lactobacillaceae: +2/-0 | Allobaculum: +2/-0 | Alistipes: +2/-0 | Bacteroides: +3/-0 | Clostridium: +1/-2 | Desulfovibrio: +0/-3 | Bacteroidota: +3/-0 | Lactobacillus: +3/-0 | Firmicutes: +1/-3 | Muribaculum: +4/-0 | Bifidobacterium: +4/-0 | Akkermansia: +5/-0

Muribaculaceae: +1/-0 (1 sin direccion) | Mucispirillum schaedleri: +0/-1 | Lactobacillus reuteri + Bifidobacterium pseudolongum: +1/-0 | Odoribacter: +1/-1 | Firmicutes: +2/-0 (1 sin direccion) | Rikenellaceae RC9 gut group: +2/-0 | Helicobacteraceae: +0/-2 | Bacteroidetes: +2/-0 (1 sin direccion) | Alistipes: +2/-0 (1 sin direccion) | Enterobacteriaceae: +0/-2 | Escherichia-Shigella: +0/-3 | Bacteroides: +3/-1 | Lactobacillus: +4/-0

